# 01: IEEE 754 Floating-Point building blocks

A 64-bit float stores: $(-1)^s \cdot 1.\text{mantissa} \cdot 2^{e - 1023}$

| segment | bits | what |
|---|---|---|
| sign | 1 | +ve or -ve |
| exponent | 11 | scale (powers of 2) |
| significand | 52 stored + 1 hidden = 53 | the actual digits |

- 53 bits of precision $\approx 15\text{-}16$ decimal digits
- Machine epsilon: $\varepsilon = 2^{-52} \approx 2.2 \times 10^{-16}$ (smallest gap at 1.0)
- Every op returns best possible 53-bit answer (often needs rounding)
- Error from one op: $\text{fl}(a \cdot b) = ab(1 + \delta)$ where $|\delta| \leq 2^{-53}$
- Problem: errors compound. Dot product of length $n$ has error $\sim n \varepsilon$

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("../src")))

In [1]:
import struct
import math
from decimal import Decimal, getcontext


def bits(label: str, x: float):
    """IEEE 754 bit printing."""
    raw = struct.unpack(">Q", struct.pack(">d", x))[0]
    sign = (raw >> 63) & 1
    exp = (raw >> 52) & 0x7FF
    frac = raw & ((1 << 52) - 1)
    sign_ch = "-" if sign else "+"

    sig_bin = f"{frac:052b}"
    grouped = " ".join(sig_bin[i : i + 4] for i in range(0, 52, 4))

    print(f" {label} = {x}")
    pad = " " * (len(label) + 2)
    print(f" {pad}{sign_ch} 1.{grouped} * 2^{exp - 1023}")

In [2]:
bits("pi", 3.141592653589793)
print()

bits("0.1", 0.1)
print()

bits("1.0", 1.0)
print()

bits("1.0 + eps", 1.0 + 2**-52)
print()

bits("1.0 + eps/2", 1.0 + 2**-53)  # rounds back to 1.0!
print()

is_equal = 1.0 + 2**-53 == 1.0
print(f"1.0 + eps/2 == 1.0? {is_equal}")
print("eps/2 is below rounding threshold, so it vanishes")

 pi = 3.141592653589793
     + 1.1001 0010 0001 1111 1011 0101 0100 0100 0100 0010 1101 0001 1000 * 2^1

 0.1 = 0.1
      + 1.1001 1001 1001 1001 1001 1001 1001 1001 1001 1001 1001 1001 1010 * 2^-4

 1.0 = 1.0
      + 1.0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 * 2^0

 1.0 + eps = 1.0000000000000002
            + 1.0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0001 * 2^0

 1.0 + eps/2 = 1.0
              + 1.0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 * 2^0

1.0 + eps/2 == 1.0? True
eps/2 is below rounding threshold, so it vanishes


## Veltkamp split

Split a 53-bit float into two $\sim 26$-bit halves.

**Why:** $26 \times 26 = 52$ bits. Fits in FP64's 53-bit significand. Partial products become exact.

**How:**
1. Multiply by $2^s + 1$ ($s = 27$) to force rounding, removes low bits
2. Subtract to recover the top bits ($a_{hi}$)
3. Leftover is $a_{lo} = a - a_{hi}$

Both halves have $\leq 26$ active significand bits. They add back exactly: $a = a_{hi} + a_{lo}$.

In [3]:
a = 3.141592653589793
s = 27

bits("a", a)
print()

# Force rounding
factor = float(2**s + 1)
c = factor * a
bits("c", c)

# Subtract to isolate high bits
cma = c - a
bits("c-a", cma)
print()

# Most significant bits
a_hi = c - cma
bits("a_hi", a_hi)

# Least significant bits
a_lo = a - a_hi
bits("a_lo", a_lo)
print()

print(f"a_hi + a_lo == a: {a_hi + a_lo == a}, no rounding error in sum")

 a = 3.141592653589793
    + 1.1001 0010 0001 1111 1011 0101 0100 0100 0100 0010 1101 0001 1000 * 2^1

 c = 421657431.40790576
    + 1.1001 0010 0001 1111 1011 0101 0111 0110 1000 0110 1100 1000 0011 * 2^28
 c-a = 421657428.2663131
      + 1.1001 0010 0001 1111 1011 0101 0100 0100 0100 0010 1101 0001 1000 * 2^28

 a_hi = 3.1415926814079285
       + 1.1001 0010 0001 1111 1011 0101 1000 0000 0000 0000 0000 0000 0000 * 2^1
 a_lo = -2.781813535079891e-08
       - 1.1101 1101 1110 1001 0111 0100 0000 0000 0000 0000 0000 0000 0000 * 2^-26

a_hi + a_lo == a: True, no rounding error in sum


## TwoProduct

Returns $(p, e)$ where $a \cdot b = p + e$ exactly. $p$ is the rounded product, $e$ is what was lost.

**Veltkamp method:** split both operands, compute 4 partial products (each exact), reconstruct error.
**FMA method:** `fma(a, b, -p)` holds the full 106-bit intermediate internally, rounds once.

In [4]:
a = 3.141592653589793
b = 2.718281828459045

bits("a", a)
bits("b", b)
print()

p = a * b
bits("p=a*b", p)

e = math.fma(a, b, -p)
bits("e=fma", e)
print()

# verify with arbitrary precision
getcontext().prec = 50
exact = Decimal(a) * Decimal(b)
recovered = Decimal(p) + Decimal(e)
print(f"p + e == a*b: {exact == recovered}")

 a = 3.141592653589793
    + 1.1001 0010 0001 1111 1011 0101 0100 0100 0100 0010 1101 0001 1000 * 2^1
 b = 2.718281828459045
    + 1.0101 1011 1111 0000 1010 1000 1011 0001 0100 0101 0111 0110 1001 * 2^1

 p=a*b = 8.539734222673566
        + 1.0001 0001 0100 0101 1000 0000 1011 0100 0101 1101 0100 0111 0100 * 2^3
 e=fma = 3.119184308355765e-16
        + 1.0110 0111 1001 1110 0001 0010 0100 1010 0110 1001 1011 0110 0000 * 2^-52

p + e == a*b: True


In [5]:
from fp_emulation.ozaki import split, two_product_fma

# verify: 26-bit halves multiply exactly (52 < 53 bit significand)
a_hi, a_lo = split(3.141592653589793)
b_hi, b_lo = split(2.718281828459045)

p = a_hi * b_hi
_, e = two_product_fma(a_hi, b_hi)

bits("a_hi", a_hi)
bits("b_hi", b_hi)
print(f"\na_hi * b_hi = {p}")
print(f"error       = {e}")

 a_hi = 3.1415926814079285
       + 1.1001 0010 0001 1111 1011 0101 1000 0000 0000 0000 0000 0000 0000 * 2^1
 b_hi = 2.7182818055152893
       + 1.0101 1011 1111 0000 1010 1000 1000 0000 0000 0000 0000 0000 0000 * 2^1

a_hi * b_hi = 8.539734226211163
error       = 0.0


## Kahan compensated summation

Naive summation accumulates $O(n \varepsilon)$ error. Kahan tracks what each addition rounds away and feeds it back. Error stays $O(\varepsilon)$ regardless of $n$.

In [6]:
import math

# e.g. harmonic series summation: 1/1 + 1/2 + ... + 1/n
# small terms added to growing sum -> rounding error accumulates
n = 1_000_000
values = [1.0 / k for k in range(1, n + 1)]

ref = math.fsum(values)  # correct rounding

# naive left-to-right
naive = 0.0
for x in values:
    naive += x

# kahan compensated
s, c = 0.0, 0.0
for x in values:
    y = x - c
    t = s + y
    c = (t - s) - y
    s = t

print(f"math.fsum: {ref:.15e}")
print(f"naive:     {naive:.15e}  err = {abs(naive - ref):.2e}")
print(f"kahan:     {s:.15e}  err = {abs(s - ref):.2e}")
print()
print(
    f"naive loses approx {-math.log10(abs(naive - ref) / ref):.0f} digits, kahan keeps full precision"
)

math.fsum: 1.439272672286572e+01
naive:     1.439272672286499e+01  err = 7.35e-13
kahan:     1.439272672286572e+01  err = 0.00e+00

naive loses ~13 digits, kahan keeps full precision


## Fixed-point Q8.8

Integer with an implicit binary point.

| property | FP64 | Q8.8 |
|---|---|---|
| error type | relative ($\sim 2^{-53}$) | absolute ($1/256$) |
| range | $\pm 10^{308}$ | $\pm 127.996$ |
| near zero | great (subnormals) | bad (same $1/256$ step) |
| hardware | expensive | just an integer ALU |

`value = raw_integer / 256`

$Q8.8 \times Q8.8 = Q16.16$ (32-bit), so accumulate in double width.

In [7]:
print("Q8.8 encoding:")
print()
for v in [1.5, -1.0, 2.25, 0.1, -42.75]:
    scale = 256
    raw = int(v * scale)
    clamped = max(-32768, min(32767, raw))
    u = clamped & 0xFFFF
    b = f"{u:016b}"
    recovered = clamped / scale
    print(f" {v:>8.4f} -> {b[:8]}.{b[8:]}  = {recovered:.4f}  err={v - recovered:+.6f}")

# 0.1 not represented exactly: 0.1 * 256 = 25.6, truncated to 25
# -1.0 is twos complement: 11111111.00000000

Q8.8 encoding:

   1.5000 -> 00000001.10000000  = 1.5000  err=+0.000000
  -1.0000 -> 11111111.00000000  = -1.0000  err=+0.000000
   2.2500 -> 00000010.01000000  = 2.2500  err=+0.000000
   0.1000 -> 00000000.00011001  = 0.0977  err=+0.002344
 -42.7500 -> 11010101.01000000  = -42.7500  err=+0.000000


In [8]:
scale = 256

a1, b1 = int(1.5 * scale), int(2.25 * scale)  # 384, 576
a2, b2 = int(-1.0 * scale), int(4.0 * scale)  # -256, 1024

print("Q8.8 MAC: 1.5*2.25 + (-1.0)*4.0")
print()
print(f" {a1} * {b1} = {a1 * b1}")
print(f" {a2} * {b2} = {a2 * b2}")
print()

acc = a1 * b1 + a2 * b2
print(f" acc = {acc} (Q16.16)")
print(f" acc / 65536 = {acc / (scale * scale)}")
print()
print("integer multiply, integer add, divide by scale^2 at the end")

Q8.8 MAC: 1.5*2.25 + (-1.0)*4.0

 384 * 576 = 221184
 -256 * 1024 = -262144

 acc = -40960 (Q16.16)
 acc / 65536 = -0.625

integer multiply, integer add, divide by scale^2 at the end
